## Hospital Pipeline: Initial Backfill (Bronze to Silver)

### Overall Objective
This notebook performs a **one-time manual backfill** to establish the initial baseline for hospital data. It ingests raw data from two distinct sources, cleans and merges them, validates data quality, applies SCD Type 2 logic to track historical changes, and persists the final result to the Silver layer (Test container) in Delta format.

### What Was Accomplished
1. **Ingested Raw Sources:** Read legacy data from a nested OneDrive JSON file and live data from the Azure SQL Bronze external table.
2. **Standardized & Cleaned:** Trimmed whitespace, cast strings to proper numeric/date types, and unified schemas across both sources.
3. **Merged & Deduplicated:** Checked for duplicate IDs using an inner join, removed redundant historical file records using a left-anti join, and unioned the unique remaining records with the live DB.
4. **Validated Quality:** Performed SQL checks for null primary keys, invalid date formats, and inconsistent timestamp logic.
5. **Applied SCD Type 2:** Used the LEAD() window function to create historical validity windows (`valid_from`, `valid_to`, `is_current`), enabling accurate change tracking.
6. **Persisted to Silver:** Wrote the final SCD2 baseline dataset as a Delta table in the Test container and registered it in the Unity Catalog.

### Next Steps
This notebook runs **ONLY ONCE**. Future daily incremental loads (Day 2 onwards) will be handled by a separate Auto Loader pipeline using Delta `MERGE` logic to update this baseline.

### 1. OneDrive Hospital File Exploring

In [0]:
file_path = "abfss://bronze@sthealthonelakehouseci.dfs.core.windows.net/onedrive_landing/hospitals_departments.json"
df_raw_file = spark.read.option("multiline", "true").json(file_path)

# Inspect schema
df_raw_file.printSchema()
# Check number of columns
print("Number of columns:", len(df_raw_file.columns))
# Look at a few records
display(df_raw_file.limit(20))


from pyspark.sql.functions import explode, col
# Explode the 'hospitals' array into separate rows (only hospitals)
df_hospitals_file = df_raw_file.select(explode(col("hospitals")).alias("hospital")).select("hospital.*")

print("Number of rows:", df_hospitals_file.count())
display(df_hospitals_file.limit(20))

# TempView
df_hospitals_file.createOrReplaceTempView("hospitals_file_view")

### 2. Data Quality Validation — hospitals_file

In [0]:
%sql
-- =============================================================================
--						Checking 'hospitals_file_view'
-- =============================================================================

/*
SELECT * 
FROM hospitals_file_view 
LIMIT 20;

-- Check 1: NULL or Duplicate Primary Keys
-- Result: No
SELECT 
    hospital_id,
    COUNT(*) AS record_count
FROM hospitals_file_view
GROUP BY hospital_id
HAVING hospital_id IS NULL
    OR COUNT(*) > 1;

-- Check 2: Checking leading or trailing whitespace in all string fields
-- Result: No
SELECT 
    hospital_id,
    name,
    address,
    city,
    region
FROM hospitals_file_view
WHERE name <> TRIM(name)
   OR address <> TRIM(address)
   OR city <> TRIM(city)
   OR region <> TRIM(region);

-- Check 3: Date Validation
-- Result: No
SELECT *
FROM (
    SELECT opened_date,
           created_at,
           updated_at,
           try_to_date(opened_date, 'yyyy-MM-dd') AS CheckDate_open_date,
           try_to_timestamp(created_at, 'yyyy-MM-dd HH:mm:ss') AS CheckDate_created_at,
           try_to_timestamp(updated_at, 'yyyy-MM-dd HH:mm:ss') AS CheckDate_updated_at
    FROM hospitals_file_view
) t
WHERE CheckDate_open_date IS NULL
   OR CheckDate_created_at IS NULL
   OR CheckDate_updated_at IS NULL
LIMIT 20;

-- Check 4: Date Logic Validation (created_at < opened_date, updated_at >= created_at)
-- Result: No
SELECT *
FROM hospitals_file_view
WHERE 
    created_at <= opened_date OR
    updated_at < created_at;
*/


### 3. Standardizing JSON File Data

In [0]:
from pyspark.sql.functions import lit, trim, col, lower, to_date, to_timestamp

# Trim and convert to lowercase for all string columns in files data
df_hospitals_file = df_hospitals_file.limit(20).select(
    col("hospital_id").cast("long"),
    lower(trim(col("name"))).alias("name"),
    lower(trim(col("address"))).alias("address"),
    lower(trim(col("city"))).alias("city"),
    lower(trim(col("region"))).alias("region"),
    col("capacity").cast("int"),
    to_date(col("opened_date"), "yyyy-MM-dd").alias("opened_date"),
    to_timestamp(col("created_at"), "yyyy-MM-dd HH:mm:ss").alias("created_at"),
    to_timestamp(col("updated_at"), "yyyy-MM-dd HH:mm:ss").alias("updated_at")
).withColumn("source", lit("file"))

display(df_hospitals_file.limit(20))

### 4. Exploring AZURE SQL DB Hospital Data

In [0]:
"""
OR
# Use spark.table to access the metadata path created in SQL (avoids writing long path again)
spark.catalog.clearCache()
df_hr_hospitals_db  = spark.table("healthone_lakehouse.bronze.hospitals")

"""

raw_path = "abfss://bronze@sthealthonelakehouseci.dfs.core.windows.net/sql-healthone-lakehouse-central-india.database.windows.net/sqldb-corporate-hr/hospitals/"
df_hr_hospitals_db = spark.read.parquet(raw_path) 

display(df_hr_hospitals_db.limit(20))
# Inspect schema
df_hr_hospitals_db.printSchema()


# TempView
df_hr_hospitals_db.createOrReplaceTempView("hr_hospitals_db_view")

### 5. Data Quality Validation — hospitals_db

In [0]:
%sql
-- =============================================================================
--						Checking 'hr_hospitals_db_view'
-- =============================================================================

/*
SELECT * 
FROM hr_hospitals_db_view 
LIMIT 20;

-- Check 1: NULL or Duplicate Primary Keys
-- Result: No
SELECT 
    hospital_id,
    COUNT(*) AS record_count
FROM hr_hospitals_db_view
GROUP BY hospital_id
HAVING hospital_id IS NULL
    OR COUNT(*) > 1;

-- Check 2: Checking leading or trailing whitespace in all string fields
-- Result: Has leading and trailling spaces
SELECT 
    hospital_id,
    name,
    address,
    city,
    region,
    source
FROM hr_hospitals_db_view
WHERE name <> TRIM(name)
   OR address <> TRIM(address)
   OR city <> TRIM(city)
   OR region <> TRIM(region);

-- Check 3: Date Validation
-- Result: No
SELECT *
FROM (
    SELECT opened_date,
           created_at,
           updated_at,
           try_to_date(opened_date, 'yyyy-MM-dd') AS CheckDate_open_date,
           try_to_timestamp(created_at, 'yyyy-MM-dd HH:mm:ss') AS CheckDate_created_at,
           try_to_timestamp(updated_at, 'yyyy-MM-dd HH:mm:ss') AS CheckDate_updated_at
    FROM hr_hospitals_db_view
) t
WHERE CheckDate_open_date IS NULL
   OR CheckDate_created_at IS NULL
   OR CheckDate_updated_at IS NULL
LIMIT 20;

-- Check 4: Date Logic Validation (created_at < opened_date, updated_at >= created_at)
-- Result: No
SELECT *
FROM hr_hospitals_db_view
WHERE 
    created_at <= opened_date OR
    updated_at < created_at;
*/


### 6. Applying SCD Type 2 logic (Initial Baseline)

In [0]:
from pyspark.sql.functions import lit, trim, col, lower, to_date, to_timestamp

# Trim and convert to lowercase for all string columns in DB data
df_hr_hospitals_db = df_hr_hospitals_db.limit(20).select(
    col("hospital_id").cast("long"),
    lower(trim(col("name"))).alias("name"),
    lower(trim(col("address"))).alias("address"),
    lower(trim(col("city"))).alias("city"),
    lower(trim(col("region"))).alias("region"),
    col("capacity").cast("int"),
    to_date(col("opened_date"), "yyyy-MM-dd").alias("opened_date"),
    to_timestamp(col("created_at"), "yyyy-MM-dd HH:mm:ss").alias("created_at"),
    to_timestamp(col("updated_at"), "yyyy-MM-dd HH:mm:ss").alias("updated_at")
).withColumn("source", lit("db"))


# Merge file and DB data
df_hospitals_combined_test = df_hospitals_file.unionByName(df_hr_hospitals_db).limit(40)
display(df_hospitals_combined_test)

# TempView
df_hospitals_combined_test.createOrReplaceTempView("hospitals_combined_test_view")

# SCD2 Apply
df_hospitals_combined_test = spark.sql("""
    SELECT
        hospital_id,
        name,
        address,
        city, 
        region, 
        capacity, 
        opened_date, 
        created_at, 
        updated_at,
        source,
        updated_at AS valid_from,
        nxt_date AS valid_to,
        CASE
            WHEN nxt_date IS NULL THEN 1
            ELSE 0
        END AS is_current
    FROM (
        SELECT
            *,
            LEAD(updated_at) OVER(
                PARTITION BY hospital_id 
                ORDER BY updated_at ASC, 
                         CASE WHEN source = 'db' THEN 1 ELSE 0 END ASC
            ) AS nxt_date
        FROM hospitals_combined_test_view
    ) t
    ORDER BY hospital_id, updated_at
""")

# Inspect schema
df_hospitals_combined_test.printSchema()
display(df_hospitals_combined_test.limit(40))

### 7. Writing to Silver Layer in Delta Format

In [0]:
target_path = "abfss://test@sthealthonelakehouseci.dfs.core.windows.net/corporate_hr/hospitals/"
table_name = "healthone_lakehouse.test.hospitals"

(df_hospitals_combined_test
    .write
    .format("delta")
    .mode("overwrite")             
    .option("overwriteSchema", "true")  
    .save(target_path)
)


# Register the delta files as a table in Unity Catalog so the team can query it via SQL
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {table_name}
    USING DELTA
    LOCATION '{target_path}'
""")

# Pack small files and group records by hospital_id
# Speeds up daily MERGE queries by skipping files that don't match the incoming IDs
spark.sql(f"""
    OPTIMIZE {table_name}
    ZORDER BY (hospital_id)
""")

display(spark.read.format("delta").load(target_path).limit(20))

In [0]:
target_path = "abfss://test@sthealthonelakehouseci.dfs.core.windows.net/corporate_hr/hospitals/"

# display(spark.sql("""SELECT * FROM healthone_lakehouse.test.hospitals"""))
display(spark.read.format("delta").load(target_path).limit(20).orderBy("hospital_id"))

### 8. Azure SQL DB Incremental Load

In [0]:
# ============================================================================
# INCREMENTAL LOAD: SILVER.HOSPITALS (SCD Type 2)
# ============================================================================
# Features:
# - Partition pruning (fast)
# - Source priority (DB wins)
# - Auto-recovery from failures
# - Watermark tracking
# - ZORDER optimization
# ============================================================================

from pyspark.sql.functions import lit, trim, col, to_date, to_timestamp, when, row_number, max as spark_max, desc, expr
from pyspark.sql import Window
from delta.tables import DeltaTable
from datetime import datetime, timedelta

# ============================================================================
# 1. CONFIGURATION
# ============================================================================

# Source bronze paths
source_base = "abfss://bronze@sthealthonelakehouseci.dfs.core.windows.net/sql-healthone-lakehouse-central-india.database.windows.net/sqldb-corporate-hr/hospitals/"

# Target silver path
target_path = "abfss://silver@sthealthonelakehouseci.dfs.core.windows.net/corporate_hr/hospitals/"
table_name = "healthone_lakehouse.silver.hospitals"

# Control/watermark table
watermark_table = "healthone_lakehouse.control.pipeline_watermark"

# ============================================================================
# 2. GET WATERMARK STATE
# ============================================================================

def get_processed_days():
    """Get list of successfully processed days from watermark table"""
    try:
        processed = spark.sql(f"""
            SELECT DISTINCT partition_path 
            FROM {watermark_table}
            WHERE table_name = 'hospitals'
              AND status = 'SUCCESS'
        """).collect()
        
        return [row.partition_path for row in processed]
    except:
        # Watermark table doesn't exist yet
        return []

def update_watermark(partition_path, rows_loaded, status, error_msg=None):
    """Update watermark table with processing status"""
    
    # Create watermark table if not exists
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {watermark_table} (
            table_name STRING,
            partition_path STRING,
            rows_loaded BIGINT,
            status STRING,
            processed_at TIMESTAMP,
            retry_count INT,
            error_message STRING
        )
        USING DELTA
    """)
    
    # Insert or update status
    spark.sql(f"""
        MERGE INTO {watermark_table} AS target
        USING (
            SELECT 
                'hospitals' AS table_name,
                '{partition_path}' AS partition_path,
                {rows_loaded} AS rows_loaded,
                '{status}' AS status,
                CURRENT_TIMESTAMP() AS processed_at,
                0 AS retry_count,
                {f"'{error_msg}'" if error_msg else 'NULL'} AS error_message
        ) AS source
        ON target.table_name = source.table_name 
           AND target.partition_path = source.partition_path
        WHEN MATCHED THEN
            UPDATE SET
                rows_loaded = source.rows_loaded,
                status = source.status,
                processed_at = source.processed_at,
                error_message = source.error_message,
                retry_count = target.retry_count + 1
        WHEN NOT MATCHED THEN
            INSERT *
    """)

# ============================================================================
# 3. GET DAYS TO PROCESS
# ============================================================================

def get_days_to_process():
    """Get list of unprocessed Day folders"""
    
    # Get all Day folders from bronze
    try:
        all_days = []
        for folder in dbutils.fs.ls(source_base):
            if folder.name.startswith("Year="):
                year = folder.name.replace("Year=", "")
                for month_folder in dbutils.fs.ls(folder.path):
                    if month_folder.name.startswith("Month="):
                        month = month_folder.name.replace("Month=", "")
                        for day_folder in dbutils.fs.ls(month_folder.path):
                            if day_folder.name.startswith("Day="):
                                day = day_folder.name.replace("Day=", "")
                                partition_path = f"Year={year}/Month={month}/Day={day}"
                                all_days.append(partition_path)
    except:
        return []
    
    # Get processed days
    processed = get_processed_days()
    
    # Return unprocessed days
    return [day for day in all_days if day not in processed]

# ============================================================================
# 4. READ NEW DATA
# ============================================================================

def read_new_data(days_to_process):
    """Read data from unprocessed Day folders"""
    
    if not days_to_process:
        print("No new days to process")
        return None
    
    df_all_new = None
    
    for day_path in days_to_process:
        try:
            # Build full path
            full_path = f"{source_base}{day_path}/"
            print(f"Processing: {day_path}")
            
            # Read ONLY this day partition (FAST!)
            df_day = (
                spark.read.format("parquet")
                .load(full_path)
                .drop("Year", "Month", "Day")
                .select(
                    col("hospital_id").cast("long"),
                    lower(trim(col("name"))).alias("name"),
                    lower(trim(col("address"))).alias("address"),
                    lower(trim(col("city"))).alias("city"),
                    lower(trim(col("region"))).alias("region"),
                    col("capacity").cast("int"),
                    to_date(col("opened_date"), "yyyy-MM-dd").alias("opened_date"),
                    to_timestamp(col("created_at"), "yyyy-MM-dd HH:mm:ss").alias("created_at"),
                    to_timestamp(col("updated_at"), "yyyy-MM-dd HH:mm:ss").alias("updated_at")
                )
                .withColumn("source", lit("db"))
                .withColumn("partition_path", lit(day_path))  # Track source
            )
            
            # Union with previous days
            if df_all_new is None:
                df_all_new = df_day
            else:
                df_all_new = df_all_new.unionByName(df_day)
                
            print(f"  ✅ Loaded {df_day.count()} rows from {day_path}")
            
        except Exception as e:
            print(f"  ❌ Failed to load {day_path}: {str(e)}")
            update_watermark(day_path, 0, "FAILED", str(e))
    
    return df_all_new

# ============================================================================
# 5. APPLY SCD TYPE 2 (MERGE LOGIC)
# ============================================================================

def apply_scd_type2(df_new_data):
    """Apply SCD Type 2 merge logic"""
    
    if df_new_data is None or df_new_data.count() == 0:
        print("No new data to process")
        return
    
    print(f"\nProcessing {df_new_data.count()} new records...")
    
    # Create temp view for new data
    df_new_data.createOrReplaceTempView("new_hospitals")
    
    # Get current records from silver
    try:
        df_current = spark.read.format("delta").load(target_path)
        df_current.createOrReplaceTempView("current_hospitals")
    except:
        # Table doesn't exist yet
        print("Silver table doesn't exist. Creating from new data...")
        df_new_data.select(
            "hospital_id", "name", "address", "city", "region", 
            "capacity", "opened_date", "created_at", "updated_at", "source",
            col("updated_at").alias("valid_from"),
            lit(None).cast("timestamp").alias("valid_to"),
            lit(1).alias("is_current")
        ).write.format("delta").mode("overwrite").save(target_path)
        
        spark.sql(f"""
            CREATE TABLE IF NOT EXISTS {table_name}
            USING DELTA
            LOCATION '{target_path}'
        """)
        
        print(f"✅ Created silver table with {df_new_data.count()} rows")
        
        # Update watermark for all processed days
        for day in df_new_data.select("partition_path").distinct().collect():
            update_watermark(day.partition_path, df_new_data.count(), "SUCCESS")
        
        return
    
    # Compare new vs current and apply SCD Type 2
    merge_result = spark.sql("""
        WITH changes AS (
            SELECT 
                new.hospital_id,
                new.name,
                new.address,
                new.city,
                new.region,
                new.capacity,
                new.opened_date,
                new.created_at,
                new.updated_at,
                new.source,
                new.partition_path,
                curr.valid_from AS current_valid_from,
                curr.is_current AS current_is_current,
                curr.capacity AS current_capacity,
                curr.name AS current_name,
                curr.address AS current_address,
                curr.city AS current_city,
                curr.region AS current_region,
                curr.opened_date AS current_opened_date
            FROM new_hospitals new
            LEFT JOIN current_hospitals curr
                ON new.hospital_id = curr.hospital_id
                AND curr.is_current = 1
        ),
        updates AS (
            SELECT hospital_id
            FROM changes
            WHERE current_is_current = 1
              AND (
                  name != current_name OR
                  address != current_address OR
                  city != current_city OR
                  region != current_region OR
                  capacity != current_capacity OR
                  opened_date != current_opened_date
              )
        ),
        inserts AS (
            SELECT 
                hospital_id,
                name,
                address,
                city,
                region,
                capacity,
                opened_date,
                created_at,
                updated_at,
                source,
                updated_at AS valid_from,
                CAST(NULL AS TIMESTAMP) AS valid_to,
                1 AS is_current,
                partition_path
            FROM changes
            WHERE current_is_current IS NULL
               OR current_is_current = 1
        )
        SELECT * FROM inserts
    """)
    
    # Close old records (updates)
    update_ids = spark.sql("""
        SELECT hospital_id FROM updates
    """).collect()
    
    if update_ids:
        update_id_list = [row.hospital_id for row in update_ids]
        print(f"Closing {len(update_id_list)} outdated records...")
        
        dt = DeltaTable.forPath(spark, target_path)
        dt.update(
            condition = col("hospital_id").isin(update_id_list) & (col("is_current") == 1),
            set = {
                "is_current": lit(0),
                "valid_to": col("updated_at")
            }
        )
    
    # Insert new records
    if merge_result.count() > 0:
        print(f"Inserting {merge_result.count()} new records...")
        merge_result.drop("partition_path").write.format("delta").mode("append").save(target_path)
    
    # Optimize table (ZORDER by hospital_id)
    spark.sql(f"OPTIMIZE {table_name} ZORDER BY (hospital_id)")
    
    print("✅ SCD Type 2 merge complete")

# ============================================================================
# 6. VERIFY RESULTS
# ============================================================================

def verify_results():
    """Verify data quality after load"""
    
    print("\n" + "="*60)
    print("VERIFICATION RESULTS")
    print("="*60)
    
    # Count total records
    total = spark.sql(f"SELECT COUNT(*) FROM {table_name}").collect()[0][0]
    current = spark.sql(f"SELECT COUNT(*) FROM {table_name} WHERE is_current = 1").collect()[0][0]
    
    print(f"Total records (history): {total}")
    print(f"Current records (active): {current}")
    
    # Check duplicates
    duplicates = spark.sql(f"""
        SELECT hospital_id, COUNT(*) as cnt
        FROM {table_name}
        WHERE is_current = 1
        GROUP BY hospital_id
        HAVING COUNT(*) > 1
    """).count()
    
    if duplicates > 0:
        print(f"⚠️ WARNING: {duplicates} duplicate hospital_ids found!")
    else:
        print("✅ No duplicate hospital_ids")
    
    # Check for NULL primary keys
    nulls = spark.sql(f"""
        SELECT COUNT(*) FROM {table_name} WHERE hospital_id IS NULL
    """).collect()[0][0]
    
    if nulls > 0:
        print(f"⚠️ WARNING: {nulls} NULL hospital_ids found!")
    else:
        print("✅ No NULL hospital_ids")

# ============================================================================
# 7. MAIN EXECUTION
# ============================================================================

print("="*60)
print("INCREMENTAL LOAD: SILVER.HOSPITALS")
print("="*60)

# Get days to process
print("\n1. Checking for unprocessed days...")
days_to_process = get_days_to_process()

if not days_to_process:
    print("✅ No new days to process")
    dbutils.notebook.exit("No new data")

print(f"📁 Found {len(days_to_process)} unprocessed days:")
for day in days_to_process:
    print(f"   - {day}")

# Read new data
print("\n2. Reading new data...")
df_new = read_new_data(days_to_process)

if df_new is None or df_new.count() == 0:
    print("No new data found")
    dbutils.notebook.exit("No data")

# Apply SCD Type 2
print("\n3. Applying SCD Type 2...")
apply_scd_type2(df_new)

# Update watermark for successful days
print("\n4. Updating watermark...")
for day in days_to_process:
    try:
        rows = df_new.filter(col("partition_path") == day).count()
        update_watermark(day, rows, "SUCCESS")
        print(f"   ✅ {day}: SUCCESS ({rows} rows)")
    except Exception as e:
        print(f"   ❌ {day}: FAILED - {str(e)}")
        update_watermark(day, 0, "FAILED", str(e))

# Verify
print("\n5. Verifying results...")
verify_results()

print("\n✅ INCREMENTAL LOAD COMPLETE!")

In [0]:
# Backfill (Day=04 to Day=07):
# Update watermark manually
spark.sql("""
    UPDATE healthone_lakehouse.control.pipeline_watermark
    SET status = 'PENDING'
    WHERE table_name = 'hospitals'
      AND partition_path IN ('Year=2026/Month=09/Day=04', '...')
""")


# Retry Failed:
# Auto-retry on next run
# Or manually:
spark.sql("""
    UPDATE healthone_lakehouse.control.pipeline_watermark
    SET status = 'PENDING'
    WHERE table_name = 'hospitals' AND status = 'FAILED'
""")